# Action / transition CDs (4 axes)

Build four coding-direction axes from the **previous-choice × upcoming-choice** 2×2 design, balanced at the cell level so each axis is orthogonalized w.r.t. the other factors.

| | Upcoming L | Upcoming R |
|---|---|---|
| **Prev L** | `L_L` (stay) | `switch_LR` |
| **Prev R** | `switch_RL` | `R_R` (stay) |

Per session we subsample the four cells to `n = min(|cell|)` trials, then fit four CDs:

| Axis | A (positive) | B (negative) |
|---|---|---|
| `prev_choice`  | Prev L (`L_L ∪ switch_LR`) | Prev R (`switch_RL ∪ R_R`) |
| `up_choice`    | Up L  (`L_L ∪ switch_RL`) | Up R  (`switch_LR ∪ R_R`) |
| `switch_stay`  | Switch (`switch_LR ∪ switch_RL`) | Stay (`L_L ∪ R_R`) |
| `switch_dir`   | `switch_LR` | `switch_RL` |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"Modules loaded from: {MODULE_PATH}")

In [ ]:
# ALM recordings — sessions of interest
sessions = [
    "ecephys_844034_2026-05-05_12-26-26_sorted_2026-05-13_16-51-38",
    "ecephys_844034_2026-05-06_12-31-42_sorted_2026-05-10_00-09-35",
    "ecephys_844034_2026-05-07_12-26-08_sorted_2026-05-10_22-15-43",
    "ecephys_844036_2026-05-04_16-06-43_sorted_2026-05-09_21-00-45",
    "ecephys_844036_2026-05-05_16-08-08_sorted_2026-05-19_17-42-51",
    "ecephys_844036_2026-05-06_16-27-46_sorted_2026-05-10_00-06-13",
]

## 1. Build the 4 balanced action/transition CDs

`build_action_transition_cds` does it all per session:
  1. read the `L_L_trials`, `switch_LR_trials`, `switch_RL_trials`, `R_R_trials` columns from each behavior CSV,
  2. subsample each of those four cells to a common `n = min(…)` (deterministic with `seed`),
  3. write the balanced cells *and* the four axis unions back into the CSV as new `*_balanced_trials` columns,
  4. invoke `build_cd_dataset` once per axis so you get four CD zarrs per session/region/window.

In [ ]:
from pathlib import Path
from ephys_dimension_reduction_CD_pipeline import (
    build_action_transition_cds,
    ACTION_AXES,
)

psth_root     = Path("/root/capsule/scratch/psth_results")
behavior_root = Path("/root/capsule/scratch/behavior_summary")
cd_root       = Path("/root/capsule/scratch/CD_results")

align                  = "go_cue"
binsize                = "0.1"
brain_regions_groups   = [[]]              # [[]] => all units
time_windows           = [(-1, 0)]          # CD-fitting window (pre-go_cue)
min_units_num          = 30
projection_time_window = None
two_fold_cv            = True
norm_mode              = "divide_sqrtN"
random_state           = 0
overwrite              = True
balance_seed           = 0   # seed for per-session 4-cell subsample
balance_suffix         = "balanced"

failed, counts = build_action_transition_cds(
    sessions=sessions,
    psth_root=psth_root,
    behavior_root=behavior_root,
    cd_root=cd_root,
    metadata=None,
    seed=balance_seed,
    suffix=balance_suffix,
    n_per_cell=None,                    # use min across cells per session
    axes=tuple(ACTION_AXES.keys()),     # all four
    overwrite_columns=True,             # rewrite CSV columns
    binsize=binsize,
    align=align,
    brain_regions_groups=brain_regions_groups,
    time_windows=time_windows,
    min_units_num=min_units_num,
    projection_time_window=projection_time_window,
    two_fold_cv=two_fold_cv,
    norm_mode=norm_mode,
    random_state=random_state,
    overwrite=overwrite,
)
print("Failed items:", failed)
print("Per-session balanced counts:", counts)

## 2. Visualize each axis per session

For every session, plot the standard 3-panel CD figure for each of the 4 axes. The CD zarr filenames carry the axis-specific column pair, so we just rebuild the path with `cd_save_path` for each axis.

In [ ]:
import os
from ephys_dimension_reduction_CD_pipeline import (
    cd_save_path,
    load_cd_session,
    plot_cd_session,
    region_label,
    ACTION_AXES,
)

viz_region_group     = []
viz_time_window      = (-1, 0)
distribution_window  = (-1, 0)
restrict_events      = ("trial_start", "go_cue")

region_lbl, region_print = region_label(viz_region_group)

# Column-name pairs that `_balance_action_cells` writes (must match the seed/suffix above).
axis_column_pairs = {
    "prev_choice":   (f"prev_L_{balance_suffix}_trials",   f"prev_R_{balance_suffix}_trials"),
    "up_choice":     (f"up_L_{balance_suffix}_trials",     f"up_R_{balance_suffix}_trials"),
    "switch_stay":   (f"switch_{balance_suffix}_trials",   f"stay_{balance_suffix}_trials"),
    "switch_dir":    (f"switch_LR_{balance_suffix}_trials", f"switch_RL_{balance_suffix}_trials"),
}

for session in sessions:
    beh_csv = os.path.join(behavior_root, f"behavior_summary-{session}.csv")
    if not os.path.exists(beh_csv):
        print(f"[skip] behavior CSV missing for {session}")
        continue
    print(f"\n=========== Session: {session} ===========")
    for axis, cols in axis_column_pairs.items():
        zpath = cd_save_path(
            cd_root, session, region_lbl, cols, viz_time_window, align=align,
        )
        if not zpath.exists():
            print(f"  [skip] {axis}: zarr not built ({zpath.name})")
            continue
        sess = load_cd_session(zpath, beh_csv)
        print(f"  --- axis = {axis} ({cols[0]} vs {cols[1]}) ---")
        plot_cd_session(
            sess,
            distribution_window=distribution_window,
            restrict_events=restrict_events,
            restrict_align=align,
            xlim=(-15, 5),
            split="test",
            smooth_gauss=0.0,
            smooth_moving_window=10,
            plot_single_trial=True,
            random_sample_trial_N=5,
        )

## 3. Project all 4 cells onto each axis

For each axis, project the four 2×2 cells (`L_L`, `switch_LR`, `switch_RL`, `R_R`) onto its CD axis. `plot_cd_session(trial_types=...)` accepts at most 2 columns at a time, so we render two figures per axis — one for stays (`L_L` vs `R_R`) and one for switches (`switch_LR` vs `switch_RL`) — together covering all four cells.

Reading the result: on the `prev_choice` axis, prev-L cells (`L_L`, `switch_LR`) should separate from prev-R cells (`switch_RL`, `R_R`) regardless of upcoming choice; on `up_choice`, the split should follow the upcoming side; etc.

In [ ]:
# Pairs of cells to overlay per axis (need 1 or 2 cols per plot_cd_session call)
cell_pairs = [
    (f"L_L_{balance_suffix}_trials",       f"R_R_{balance_suffix}_trials"),        # stays
    (f"switch_LR_{balance_suffix}_trials", f"switch_RL_{balance_suffix}_trials"),  # switches
]

for session in sessions:
    beh_csv = os.path.join(behavior_root, f"behavior_summary-{session}.csv")
    if not os.path.exists(beh_csv):
        continue
    print(f"\n=========== Session: {session} ===========")
    for axis, cols in axis_column_pairs.items():
        zpath = cd_save_path(
            cd_root, session, region_lbl, cols, viz_time_window, align=align,
        )
        if not zpath.exists():
            continue
        sess = load_cd_session(zpath, beh_csv)
        for pair_name, pair_cols in zip(("stays", "switches"), cell_pairs):
            print(f"  axis = {axis} | {pair_name}: {pair_cols[0]} vs {pair_cols[1]}")
            plot_cd_session(
                sess,
                distribution_window=distribution_window,
                restrict_events=restrict_events,
                restrict_align=align,
                xlim=(-15, 5),
                split="test",
                smooth_gauss=0.0,
                smooth_moving_window=10,
                plot_single_trial=True,
                random_sample_trial_N=5,
                trial_types=list(pair_cols),
            )

## 4. Decode trial types from the CD projection

For every axis, fit a 1D **decision boundary** on the train-fold projections (averaged over the decoding window) and evaluate it on the **held-out test** trials. We use the balanced-accuracy-optimal threshold and report:

- training & test accuracy + balanced accuracy
- ROC curve + AUC on test trials
- 2x2 confusion matrix
- accuracy & AUC as a function of time (sliding window) so you can see *when* each variable becomes decodable.

In [ ]:
from ephys_dimension_reduction_CD_pipeline import (
    decode_cd_session,
    plot_cd_decoder,
)

# Window over which to average the projection trace into one scalar/trial.
decode_window = (-1.0, 0.0)          # same as CD-fitting window
decode_method = "optimal"             # "optimal" | "midpoint"
time_curve_bin = 0.2                  # sliding decoder window (s)

# Collect per-axis / per-session accuracy for the summary plot.
import pandas as pd
decode_rows = []

for session in sessions:
    beh_csv = os.path.join(behavior_root, f"behavior_summary-{session}.csv")
    if not os.path.exists(beh_csv):
        continue
    print(f"\n=========== Session: {session} ===========")
    for axis, cols in axis_column_pairs.items():
        zpath = cd_save_path(
            cd_root, session, region_lbl, cols, viz_time_window, align=align,
        )
        if not zpath.exists():
            continue
        sess = load_cd_session(zpath, beh_csv)
        print(f"  --- axis = {axis} ({cols[0]} vs {cols[1]}) ---")
        result = plot_cd_decoder(
            sess,
            window=decode_window,
            method=decode_method,
            show_time_curve=True,
            time_curve_bin=time_curve_bin,
        )
        decode_rows.append({
            "session": session,
            "axis": axis,
            "boundary": result.boundary,
            "train_acc": result.train_accuracy,
            "test_acc": result.test_accuracy,
            "train_bal_acc": result.train_balanced_accuracy,
            "test_bal_acc": result.test_balanced_accuracy,
            "test_auc": result.test_auc,
            "n_test_A": int(result.test_scores_a.size),
            "n_test_B": int(result.test_scores_b.size),
        })

decode_df = pd.DataFrame(decode_rows)
decode_df

### Summary across sessions

Bar plot of the test accuracy & AUC per axis (dots = individual sessions, bar = mean).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if len(decode_df):
    axes_order = list(axis_column_pairs.keys())
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True)
    for ax, metric, ylabel in zip(
        axs,
        ("test_bal_acc", "test_auc"),
        ("Test balanced accuracy", "Test AUC"),
    ):
        xs = np.arange(len(axes_order))
        means = [decode_df.loc[decode_df.axis == a, metric].mean() for a in axes_order]
        ax.bar(xs, means, color="tab:gray", alpha=0.6, edgecolor="k")
        for i, a in enumerate(axes_order):
            vals = decode_df.loc[decode_df.axis == a, metric].values
            jitter = (np.random.default_rng(0).uniform(-0.15, 0.15, size=vals.size))
            ax.scatter(np.full_like(vals, i, dtype=float) + jitter, vals,
                       color="tab:blue", s=40, edgecolor="k", zorder=3)
        ax.axhline(0.5, color="red", linestyle=":", lw=1, label="chance")
        ax.set_xticks(xs)
        ax.set_xticklabels(axes_order, rotation=15, ha="right")
        ax.set_ylim(0.3, 1.0)
        ax.set_ylabel(ylabel)
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend(fontsize=9)
    fig.suptitle("CD decoder — per-axis test performance (each dot = session)")
    fig.tight_layout()
    plt.show()
else:
    print("No decoder results to summarize.")

## 5. Time-resolved decoding (CD re-fit per time bin)

For every time bin (centered at `c`, width `bin_window`):

1. Find trials whose `[restrict_events[0], restrict_events[1]]` interval (offsets
   relative to `restrict_align`) fully covers `[c - bin_window/2, c + bin_window/2]`.
2. Intersect the 4 action cells (`L_L`, `switch_LR`, `switch_RL`, `R_R`) with this
   eligible set and subsample each to `n = min(...)` ? so all four axes stay
   balanced w.r.t. the orthogonal factors at this bin.
3. Fit a fresh CD axis for each axis on the balanced cell unions, using only this
   bin's data window, and decode the held-out trials with a 1D threshold.

The result is one accuracy / AUC curve per axis. The drop near long pre-trial
times reflects fewer eligible trials (some sessions have shorter ITIs); the
secondary axis shows the minimum balanced sample size per bin across sessions.


In [ ]:
from ephys_dimension_reduction_CD_pipeline import (
    decode_action_axes_over_time,
    plot_action_decoding_over_time,
)
import numpy as np

# Time-bin grid (relative to PSTH ``align``, here go_cue)
tr_t_start = -2.0
tr_t_end   =  0.5
tr_step    =  0.1     # how often to fit a new CD axis
tr_window  =  0.2     # width of each CD-fit + decoding window

# Trial eligibility: only use trials whose [trial_start, go_cue] covers the bin
tr_restrict_events = ("trial_start", "go_cue")
tr_restrict_align  = "go_cue"

tr_df = decode_action_axes_over_time(
    sessions=sessions,
    psth_root=psth_root,
    behavior_root=behavior_root,
    metadata=None,                # all units
    binsize=binsize,
    align=align,
    t_start=tr_t_start,
    t_end=tr_t_end,
    bin_step=tr_step,
    bin_window=tr_window,
    restrict_events=tr_restrict_events,
    restrict_align=tr_restrict_align,
    axes=("prev_choice", "up_choice", "switch_stay", "switch_dir"),
    region_group=(),
    min_units_num=min_units_num,
    n_per_cell=None,              # auto = min across cells per bin
    seed=0,
    norm_mode=norm_mode,
    decoder_method="optimal",
    two_fold_cv=True,
    random_state=random_state,
    verbose=True,
)
tr_df.head()


In [ ]:
# Mean +/- SEM across sessions
plot_action_decoding_over_time(
    tr_df, metric="test_bal_acc", aggregate="mean_sem", show_n=True,
)

# Per-session thin lines + group mean
plot_action_decoding_over_time(
    tr_df, metric="test_auc", aggregate="sessions", show_n=False,
)
